# 可选实验：特征缩放和学习率（多变量）

## 目标
在这个实验中，你将：
- 利用上一个实验开发的多变量例程
- 在具有多个特征的数据集上运行梯度下降
- 探索 *学习率 alpha* 对梯度下降的影响
- 通过使用 z-score 标准化进行 *特征缩放* 来提高梯度下降的性能

## 工具
你将使用上一个实验开发的函数以及 matplotlib 和 NumPy。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import  load_house_data, run_gradient_descent 
from lab_utils_multi import  norm_plot, plt_equal_scale, plot_cost_i_w
from lab_utils_common import dlc
np.set_printoptions(precision=2)
plt.style.use('./deeplearning.mplstyle')

## 符号

|通用 <br />  符号  | 描述| Python (如果适用) |
|: ------------|: ------------------------------------------------------------||
| $a$ | 标量，非粗体                                                      ||
| $\mathbf{a}$ | 向量，粗体                                                 ||
| $\mathbf{A}$ | 矩阵，粗体大写                                         ||
| **回归** |         |    |     |
|  $\mathbf{X}$ | 训练样本矩阵                  | `X_train` |   
|  $\mathbf{y}$  | 训练样本目标                | `y_train` 
|  $\mathbf{x}^{(i)}$, $y^{(i)}$ | 第 $i$ 个训练样本 | `X[i]`, `y[i]`|
| m | 训练样本数量 | `m`|
| n | 每个样本的特征数量 | `n`|
|  $\mathbf{w}$  |  参数：权重                       | `w`    |
|  $b$           |  参数：偏置                                           | `b`    |     
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 模型在 $\mathbf{x}^{(i)}$ 处的评估结果，由 $\mathbf{w},b$ 参数化：$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$  | `f_wb` | 
|$\frac{\partial J(\mathbf{w},b)}{\partial w_j}$| 代价关于参数 $w_j$ 的梯度或偏导数 |`dj_dw[j]`| 
|$\frac{\partial J(\mathbf{w},b)}{\partial b}$| 代价关于参数 $b$ 的梯度或偏导数| `dj_db`|

# 问题陈述

正如在之前的实验中，你将使用房价预测的激励示例。训练数据集包含许多示例，具有 4 个特征（大小、卧室、楼层和年龄），如下表所示。注意，在这个实验中，大小特征是平方英尺，而早期的实验使用的是 1000 平方英尺。这个数据集比以前的实验更大。

我们想使用这些值建立一个线性回归模型，以便我们可以预测其他房屋的价格 - 例如，一个 1200 平方英尺，3 间卧室，1 层，40 年房龄的房子。

## 数据集：
| 面积 (平方英尺) | 卧室数量  | 楼层数量 | 房屋年龄 | 价格 (1000美元)  |   
| ----------------| ------------------- |----------------- |--------------|----------------------- |  
| 952             | 2                   | 1                | 65           | 271.5                  |  
| 1244            | 3                   | 2                | 64           | 232                    |  
| 1947            | 3                   | 2                | 17           | 509.8                  |  
| ...             | ...                 | ...              | ...          | ...                    |


In [ ]:
# load the dataset
X_train, y_train = load_house_data()
X_features = ['size(sqft)','bedrooms','floors','age']

让我们通过绘制每个特征与价格的关系图来查看数据集及其特征。

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000's)")
plt.show()

绘制每个特征与目标（价格）的关系图，可以提供一些关于哪些特征对价格影响最大的指示。在上面，增加面积也会增加价格。卧室和楼层似乎对价格没有太大影响。新房子的价格比老房子高。

<a name="toc_15456_5"></a>
## 多变量梯度下降
这是你在上一个实验中开发的关于多变量梯度下降的方程：

$$\begin{align*} \text{重复}&\text{ 直到收敛:} \; \lbrace \newline\;
& w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j = 0..n-1}\newline
&b\ \ := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

其中，n 是特征数量，参数 $w_j$, $b$ 是同时更新的，其中

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3}
\end{align}
$$
* m 是数据集中的训练样本数量

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测，而 $y^{(i)}$ 是目标值


## 学习率
<figure>
    <img src="./images/C1_W2_Lab06_learningrate.png" style="width:1200px;" >
</figure>
讲座讨论了与设置学习率 $\alpha$ 有关的一些问题。学习率控制参数更新的大小。见上面的方程 (1)。它由所有参数共享。

让我们运行梯度下降并在我们的数据集上尝试 $\alpha$ 的一些设置

### $\alpha$ = 9.9e-7

In [ ]:
#set alpha to 9.9e-7
_, _, hist = run_gradient_descent(X_train, y_train, 10, alpha = 9.9e-7)

似乎学习率太高了。解不收敛。代价在 *增加* 而不是减少。让我们绘制结果：

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

右边的图显示了参数之一 $w_0$ 的值。在每次迭代中，它都超过了最优值，结果代价最终 *增加* 而不是接近最小值。注意，这并不是完全准确的图片，因为每次通过都有 4 个参数被修改，而不仅仅是一个。此图仅显示 $w_0$，其他参数固定在良性值。在此图和后续图中，你可能会注意到蓝色和橙色线略有偏差。


### $\alpha$ = 9e-7
让我们尝试一个小一点的值，看看会发生什么。

In [ ]:
#set alpha to 9e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 9e-7)

代价在整个运行过程中都在下降，表明 alpha 不太高。

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

在左边，你看到代价正如预期的那样下降。在右边，你可以看到 $w_0$ 仍然在最小值周围振荡，但每次迭代它都在减小而不是增加。注意上面 `dj_dw[0]` 随着 `w[0]` 跳过最优值而每次迭代改变符号。
这个 alpha 值将收敛。你可以改变迭代次数来看看它的表现。

### $\alpha$ = 1e-7
让我们尝试一个更小的 $\alpha$ 值，看看会发生什么。

In [ ]:
#set alpha to 1e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 1e-7)

代价在整个运行过程中都在下降，表明 $\alpha$ 不太高。

In [ ]:
plot_cost_i_w(X_train,y_train,hist)

在左边，你看到代价正如预期的那样下降。在右边你可以看到 $w_0$ 在没有越过最小值的情况下下降。注意上面 `dj_w0` 在整个运行过程中都是负的。这个解也会收敛，虽然不像前面的例子那么快。

## 特征缩放
<figure>
    <img src="./images/C1_W2_Lab06_featurescalingheader.png" style="width:1200px;" >
</figure>
讲座描述了重新缩放数据集的重要性，以便特征具有相似的范围。
如果你对为什么会这样感兴趣，请点击下面的“详情”标题。如果不想，下面的部分将逐步介绍如何进行特征缩放的实现。

<details>
<summary>
    <font size='3', color='darkgreen'><b>详情</b></font>
</summary>

让我们再次看看 $\alpha$ = 9e-7 的情况。这非常接近我们可以设置的 $\alpha$ 的最大值而不发散。这是一个显示前几次迭代的短运行：

<figure>
    <img src="./images/C1_W2_Lab06_ShortRun.png" style="width:1200px;" >
</figure>

在上面，虽然代价在减少，但很明显 $w_0$ 比其他参数进展得更快，因为它的梯度大得多。

下面的图表显示了 $\alpha$ = 9e-7 的非常长运行的结果。这需要几个小时。

<figure>
    <img src="./images/C1_W2_Lab06_LongRun.png" style="width:1200px;" >
</figure>
    
在上面，你可以看到代价在初始减少后缓慢下降。注意 `w0` 和 `w1`,`w2`,`w3` 以及 `dj_dw0` 和 `dj_dw1-3` 之间的区别。`w0` 非常快地达到其接近最终值，并且 `dj_dw0` 迅速减少到一个小值，表明 `w0` 接近最终值。其他参数减少得慢得多。

这是为什么？有什么我们可以改进的吗？见下文：
<figure>
    <center> <img src="./images/C1_W2_Lab06_scale.png"   ></center>
</figure>   

上图显示了为什么 $w$ 更新不均匀。
- $\alpha$ 由所有参数更新（$w$ 和 $b$）共享。
- 公共误差项乘以 $w$ 的特征。（不是 $b$）。
- 特征在幅度上差异很大，使得某些特征比其他特征更新得快得多。在这种情况下，$w_0$ 乘以 '面积(平方英尺)'，通常 > 1000，而 $w_1$ 乘以 '卧室数量'，通常为 2-4。
    
解决方案是特征缩放。

讲座讨论了三种不同的技术：
- 特征缩放，本质上是将每个正特征除以其最大值，或者更一般地，使用 (x-min)/(max-min) 通过其最小值和最大值重新缩放每个特征。这两种方法都将特征标准化为 -1 到 1 的范围，前一种方法适用于正特征，简单且适用于讲座的示例，后一种方法适用于任何特征。
- 均值归一化：$x_i := \dfrac{x_i - \mu_i}{max - min} $ 
- Z-score 标准化，我们将探讨如下。


### z-score 标准化
在 z-score 标准化之后，所有特征的均值都将为 0，标准差为 1。

要实现 z-score 标准化，请按此公式调整输入值：
$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{4}$$ 
其中 $j$ 选择 $\mathbf{X}$ 矩阵中的特征或列。$µ_j$ 是特征 (j) 的所有值的均值，$\sigma_j$ 是特征 (j) 的标准差。
$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{5}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{6}
\end{align}
$$

>**实现说明：**在标准化特征时，重要的是要存储用于标准化的值 - 用于计算的均值和标准差。从模型中学习参数后，我们经常想要预测以前没见过的房屋价格。给定一个新的 x 值（客厅面积和卧室数量），我们必须首先使用我们之前从训练集中计算出的均值和标准差来标准化 x。

**实现**

In [ ]:
def zscore_normalize_features(X):
    """
    computes  X, zcore normalized by column
    
    Args:
      X (ndarray (m,n))     : input data, m examples, n features
      
    Returns:
      X_norm (ndarray (m,n)): input normalized by column
      mu (ndarray (n,))     : mean of each feature
      sigma (ndarray (n,))  : standard deviation of each feature
    """
    # find the mean of each column/feature
    mu     = np.mean(X, axis=0)                 # mu will have shape (n,)
    # find the standard deviation of each column/feature
    sigma  = np.std(X, axis=0)                  # sigma will have shape (n,)
    # element-wise, subtract mu for that column from each example, divide by std for that column
    X_norm = (X - mu) / sigma      

    return (X_norm, mu, sigma)
 
#check our work
#from sklearn.preprocessing import scale
#scale(X_orig, axis=0, with_mean=True, with_std=True, copy=True)

让我们看看 Z-score 标准化涉及的步骤。下图逐步显示了转换。

In [ ]:
mu     = np.mean(X_train,axis=0)   
sigma  = np.std(X_train,axis=0) 
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma      

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[0].set_title("unnormalized")
ax[0].axis('equal')

ax[1].scatter(X_mean[:,0], X_mean[:,3])
ax[1].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[2].set_title(r"Z-score normalized")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("distribution of features before, during, after normalization")
plt.show()

上图显示了两个训练集参数“年龄”和“面积(平方英尺)”之间的关系。*这些是以相同的比例绘制的*。
- 左：未标准化：'面积(平方英尺)' 特征的值范围或方差远大于年龄
- 中：第一步从每个特征中减去均值或平均值。这留下了以零为中心的特征。很难看到“年龄”特征的差异，但“面积(平方英尺)”显然在零附近。
- 右：第二步除以方差。这使得两个特征都以零为中心，具有相似的比例。

让我们标准化数据并将其与原始数据进行比较。

In [ ]:
# normalize the original features
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print(f"X_mu = {X_mu}, \nX_sigma = {X_sigma}")
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X_train,axis=0)}")   
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X_norm,axis=0)}")

通过标准化，每列的峰峰值范围从数千倍减少到 2-3 倍。

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3))
for i in range(len(ax)):
    norm_plot(ax[i],X_train[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count");
fig.suptitle("distribution of features before normalization")
plt.show()
fig,ax=plt.subplots(1,4,figsize=(12,3))
for i in range(len(ax)):
    norm_plot(ax[i],X_norm[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count"); 
fig.suptitle("distribution of features after normalization")

plt.show()

注意，在上面，标准化数据的范围（x 轴）以零为中心，大约为 +/- 2。最重要的是，每个特征的范围相似。

让我们用标准化数据重新运行我们的梯度下降算法。
注意 **alpha 的值大得多**。这将加速梯度下降。

In [ ]:
w_norm, b_norm, hist = run_gradient_descent(X_norm, y_train, 1000, 1.0e-1, )

缩放后的特征得到非常准确的结果 **快得多！**。注意在这个相当短的运行结束时，每个参数的梯度都很小。0.1 的学习率对于标准化特征的回归是一个很好的开始。
让我们绘制我们的预测与目标值的关系图。注意，预测是使用标准化特征进行的，而图表是使用原始特征值显示的。

In [ ]:
#predict target using normalized features
m = X_norm.shape[0]
yp = np.zeros(m)
for i in range(m):
    yp[i] = np.dot(X_norm[i], w_norm) + b_norm

    # plot predictions and targets versus original features    
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color=dlc["dlorange"], label = 'predict')
ax[0].set_ylabel("Price"); ax[0].legend();
fig.suptitle("target versus prediction using z-score normalized model")
plt.show()

结果看起来不错。有几点需要注意：
- 对于多个特征，我们不再能用一个图显示结果与特征的关系。
- 生成图表时，使用了标准化特征。使用从标准化训练集学到的参数进行的任何预测也必须标准化。

**预测**
生成我们模型的目的是用它来预测不在数据集中的房价。让我们预测一个 1200 平方英尺，3 间卧室，1 层，40 年房龄的房子的价格。回想一下，你必须使用在标准化训练数据时得出的均值和标准差来标准化数据。

In [ ]:
# First, normalize out example.
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
print(x_house_norm)
x_house_predict = np.dot(x_house_norm, w_norm) + b_norm
print(f" predicted price of a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old = ${x_house_predict*1000:0.0f}")

**代价等高线**  
<img align="left" src="./images/C1_W2_Lab06_contours.png"   style="width:240px;" >另一种看待特征缩放的方法是代价等高线。当特征比例不匹配时，等高线图中的代价与参数的图是不对称的。

在下图中，参数的比例是匹配的。左图是标准化特征之前 w[0]（平方英尺）与 w[1]（卧室数量）的代价等高线图。该图非常不对称，完成等高线的曲线不可见。相反，当特征标准化时，代价等高线更加对称。结果是，梯度下降期间参数的更新可以使每个参数取得相同的进展。


In [ ]:
plt_equal_scale(X_train, X_norm, y_train)


## 恭喜！
在这个实验中，你：
- 利用了你在以前的实验中开发的多特征线性回归例程
- 探索了学习率 $\alpha$ 对收敛的影响
- 发现了使用 z-score 标准化进行特征缩放于加速收敛的价值

## 致谢
住房数据源自 Dean De Cock 编写的 [Ames Housing 数据集](http://jse.amstat.org/v19n3/decock.pdf)，用于数据科学教育。